# 🧠 BERTopic — Virality & Retweet Topic Modeling

End-to-end pipeline for discovering topics that drive retweet engagement.

**Pipeline overview:**
1. Load & inspect data
2. Clean text
3. Define embedding model
4. Compute & cache embeddings
5. Define representation models (KeyBERT, MMR, POS, OpenAI)
6. Define UMAP, HDBSCAN, Vectorizer
7. Train BERTopic weighted by retweets
8. Evaluate & inspect topics
9. Visualise results

## 📂 1. Load Data

In [13]:
import sqlite3
import pandas as pd
from datetime import datetime
from tabulate import tabulate

DB_PATH = "../data/tweetsCleanedDB.sqlite"
TABLE_NAME = "tweets"

conn = sqlite3.connect(DB_PATH)

print(f"Processing table: {TABLE_NAME}")

cols = pd.read_sql_query(f"PRAGMA table_info({TABLE_NAME});", conn)["name"].tolist()

df = pd.read_sql(f"SELECT * FROM {TABLE_NAME}", conn)
print(f"Loaded {len(df)} rows")

if 'Timestamp' in df.columns:
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
    df['date'] = df['Timestamp'].dt.date
    df['time'] = df['Timestamp'].dt.time
    df['day_of_week'] = df['Timestamp'].dt.day_name()
    df['hour'] = df['Timestamp'].dt.hour
    df['year'] = df['Timestamp'].dt.year

print(f"\n{'='*50}")
print(f"Columns: {df.columns.tolist()}")
summary = pd.DataFrame([(TABLE_NAME, len(df))], columns=["table_name", "num_rows"])
print(tabulate(summary, headers="keys", tablefmt="github", showindex=False))

# Column datatypes
print(f"\n{'='*50}")
print("Column Datatypes:")
dtype_df = pd.DataFrame({
    "column": df.dtypes.index,
    "pandas_dtype": df.dtypes.values.astype(str),
    "sqlite_dtype": pd.read_sql_query(f"PRAGMA table_info({TABLE_NAME});", conn)["type"].tolist() + ["derived"] * (len(df.columns) - len(cols))
})
print(tabulate(dtype_df, headers="keys", tablefmt="github", showindex=False))

Processing table: tweets
Loaded 9909 rows

Columns: ['tweet_id', 'username', 'text', 'retweets', 'likes', 'timestamp', 'date', 'time', 'hour', 'day_name', 'year_week', 'year_month', 'year']
| table_name   |   num_rows |
|--------------|------------|
| tweets       |       9909 |

Column Datatypes:
| column     | pandas_dtype   | sqlite_dtype   |
|------------|----------------|----------------|
| tweet_id   | object         | TEXT           |
| username   | object         | TEXT           |
| text       | object         | TEXT           |
| retweets   | int64          | INTEGER        |
| likes      | int64          | INTEGER        |
| timestamp  | object         | TIMESTAMP      |
| date       | object         | DATE           |
| time       | object         | TEXT           |
| hour       | int64          | INTEGER        |
| day_name   | object         | TEXT           |
| year_week  | object         | TEXT           |
| year_month | object         | TEXT           |
| year       | 

## 📦 2. Imports & Setup

In [14]:
import os
import re
import numpy as np
import pandas as pd
import torch
import openai

from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech

print('✅ All imports successful.')

✅ All imports successful.


## 🧹 3. Text Cleaning

Before computing embeddings, the raw text is cleaned to remove noise that could degrade topic quality.

The following transformations are applied:

- **Lowercase** — normalises casing so "Twitter" and "twitter" are treated identically
- **URLs removed** — `http://...` and `www....` links carry no semantic meaning for topic modeling
- **Mentions removed** — `@username` tags are user-specific and not topically informative
- **Hashtags normalised** — `#` symbol stripped but the word is kept (e.g. `#AI` → `AI`)
- **RT prefix removed** — retweet markers (`RT`) are artefacts of the platform, not content
- **Special characters removed** — punctuation and symbols are stripped; only letters, numbers, and spaces retained
- **Whitespace collapsed** — multiple spaces and newlines are reduced to a single space

> ⚠️ **Note:** The character filter `[^a-zA-Z0-9\s]` removes non-Latin characters. If your dataset contains non-English text (Arabic, Chinese, etc.), change the filter to `[^\w\s]` to preserve unicode word characters.

In [15]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)           # remove URLs
    text = re.sub(r'@\w+', '', text)                      # remove mentions
    text = re.sub(r'#(\w+)', r'\1', text)                 # strip # but keep word
    text = re.sub(r'^rt\s+', '', text, flags=re.MULTILINE)# remove RT prefix
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)           # remove special chars
    text = re.sub(r'\s+', ' ', text).strip()              # collapse whitespace
    return text

df['text_clean'] = df['text'].apply(clean_text)
df = df[df['text_clean'].str.len() > 0].reset_index(drop=True)

docs = df['text_clean'].tolist()
print(f'✅ Cleaned {len(docs)} documents.')
print(f'Sample: {docs[0]}')

✅ Cleaned 9909 documents.
Sample: party least receive say or single prevent prevent husband affect may himself cup style evening protect effect another themselves stage perform possible try tax share style television with successful much sell development economy effect


## 🔡 4. Embedding Model

In [16]:
EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'

# Auto-detect best available device
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'▶ Using device: {device}')

embedding_model = SentenceTransformer(EMBED_MODEL, device=device)

if device == 'cuda':
    embedding_model = embedding_model.half()  # FP16: ~2x throughput, half VRAM

▶ Using device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 🔢 5. Compute & Cache Embeddings

In [17]:
CACHE_PATH = 'embeddings_cache.npy'

if os.path.exists(CACHE_PATH):
    print(f"Loading cached embeddings from '{CACHE_PATH}'...")
    embeddings = np.load(CACHE_PATH)
    print(f'✅ Loaded from cache — shape: {embeddings.shape}')
else:
    print('Computing embeddings...')
    batch_size = 256 if device == 'cuda' else (64 if device == 'mps' else 32)

    embeddings = embedding_model.encode(
        docs,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
        device=device,
    )

    np.save(CACHE_PATH, embeddings)
    print(f"✅ Done — shape: {embeddings.shape}. Cached to '{CACHE_PATH}'")

Loading cached embeddings from 'embeddings_cache.npy'...
✅ Loaded from cache — shape: (9909, 384)


## ✂️ 3b. Train-Test Split

Split data into 80% train and 20% test **after** embeddings are loaded, so the full embedding cache is reused and only the training slice is passed to BERTopic.

- BERTopic is trained **only** on `train` documents and embeddings.
- The `test` split is held out entirely and used only for evaluation.
- Stratification on `popularity_label` ensures balanced class representation in both splits.

In [18]:
from sklearn.model_selection import train_test_split

# ── Random 80/20 split ────────────────────────────────────────────────────────
train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
)

df_train = df.loc[train_idx].reset_index(drop=True)
df_test  = df.loc[test_idx].reset_index(drop=True)

# ── Corresponding document lists ──────────────────────────────────────────────
docs_train = df_train['text_clean'].tolist()
docs_test  = df_test['text_clean'].tolist()

# ── Corresponding embeddings slices ───────────────────────────────────────────
embeddings_train = embeddings[train_idx]
embeddings_test  = embeddings[test_idx]

print(f'✅ Train size : {len(df_train):>5} docs  ({len(df_train)/len(df)*100:.1f}%)')
print(f'   Test  size : {len(df_test):>5} docs  ({len(df_test)/len(df)*100:.1f}%)')

✅ Train size :  7927 docs  (80.0%)
   Test  size :  1982 docs  (20.0%)


## 🎭 5. Representation Models

Defines fast keyword-based models for the training hot path, and an optional OpenAI model applied as a deferred update step after fitting.

In [19]:
# ── Fast keyword models (used during fit_transform) ───────────────────────────
keybert_model = KeyBERTInspired(top_n_words=5)
mmr_model     = MaximalMarginalRelevance(diversity=0.3, top_n_words=5)
pos_model     = PartOfSpeech(
    'en_core_web_sm',
    pos_patterns=[
        [{'POS': 'NOUN'}],
        [{'POS': 'ADJ'}],
        [{'POS': 'ADJ'}, {'POS': 'NOUN'}],
    ]
)

fast_representation = {
    'KeyBERT': keybert_model,
    'MMR':     mmr_model,
    'POS':     pos_model,
}

# ── OpenAI model (deferred — applied after fit_transform) ─────────────────────
prompt = """
Documents: [DOCUMENTS]
Keywords: [KEYWORDS]

Give a 5-word max label for this topic. Focus on virality drivers: emotion,
controversy, humor, or trends. Mark high-engagement topics with strong wording,
low-engagement with neutral wording.

Output: topic: <label>
"""

try:
    with open('../keys/openai-key.txt', 'r') as f:
        openai_api_key = f.read().strip()
    client = openai.OpenAI(api_key=openai_api_key)
    print('✅ OpenAI API key loaded.')
except FileNotFoundError:
    print('⚠️  Key file not found — OpenAI labelling will be skipped.')
    client = None
except Exception as e:
    print(f'⚠️  Error loading key: {e}')
    client = None

if client:
    openai_model = OpenAI(
        client,
        model='gpt-4o-mini',
        exponential_backoff=True,
        chat=True,
        prompt=prompt,
        nr_docs=3,
        delay_in_seconds=0.5,
    )
    representation_model = {**fast_representation, 'OpenAI': openai_model}
else:
    openai_model = None
    representation_model = fast_representation

print(f'Representation models: {list(representation_model.keys())}')

✅ OpenAI API key loaded.
Representation models: ['KeyBERT', 'MMR', 'POS', 'OpenAI']


## ⚙️ 6. UMAP, HDBSCAN & Vectorizer

In [20]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    low_memory=True,
    random_state=42,
    n_jobs=-1,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
    approx_min_span_tree=True,
    core_dist_n_jobs=-1,
)

vectorizer_model = CountVectorizer(
    max_features=10_000,
    min_df=3,
    ngram_range=(1, 2),
    strip_accents='unicode',
)

ctfidf_model = ClassTfidfTransformer(
    bm25_weighting=True,
    reduce_frequent_words=True,
)

print('✅ Pipeline components defined.')

✅ Pipeline components defined.


## 🏋️ 7. Train BERTopic

The model is trained with retweet-based engagement weights passed as `y`, biasing keyword extraction toward high-virality documents. The OpenAI labelling step is deferred to after fitting to keep the training loop fast.

In [21]:
# ── Compute retweet-based engagement weight ───────────────────────────────────
df_train['engagement_weight'] = np.log1p(df_train['retweets'])
df_train['engagement_weight'] = (
    df_train['engagement_weight'] - df_train['engagement_weight'].min()
) / (df_train['engagement_weight'].max() - df_train['engagement_weight'].min())

print(f"✅ Retweet weights computed (train only). Range: "
      f"{df_train['engagement_weight'].min():.3f} – {df_train['engagement_weight'].max():.3f}")

# ── Initialise BERTopic ───────────────────────────────────────────────────────
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=fast_representation,
    top_n_words=10,
    verbose=True,
    calculate_probabilities=False,
)

# ── Fit ───────────────────────────────────────────────────────────────────────
topics_train, _ = topic_model.fit_transform(
    docs_train,
    embeddings=embeddings_train,
    y=df_train['engagement_weight'].values,
)

df_train['topic_id'] = topics_train

n_topics = len(set(topics_train)) - (1 if -1 in topics_train else 0)
n_noise  = (np.array(topics_train) == -1).sum()
print(f'\n✅ Training complete.')
print(f'   Topics found   : {n_topics}')
print(f'   Noise docs (-1): {n_noise} ({100 * n_noise / len(topics_train):.1f}%)')
print('\n⚠️  Model trained on TRAIN split only — test split held out for evaluation.')

2026-03-21 12:11:13,711 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


✅ Retweet weights computed (train only). Range: 0.000 – 1.000


2026-03-21 12:12:09,750 - BERTopic - Dimensionality - Completed ✓
2026-03-21 12:12:09,752 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-21 12:12:10,392 - BERTopic - Cluster - Completed ✓
2026-03-21 12:12:10,400 - BERTopic - Representation - Extracting topics from clusters using representation models.
2026-03-21 12:13:57,427 - BERTopic - Representation - Completed ✓



✅ Training complete.
   Topics found   : 42
   Noise docs (-1): 5361 (67.6%)

⚠️  Model trained on TRAIN split only — test split held out for evaluation.


## 🔤 8. Deferred Lamma Labelling (Async Parallel)

Replaces the slow sequential `update_topics` call with fully parallel async requests.
All 45 topics are labelled concurrently — target runtime under 30 seconds.

**Why this is fast:**
- `asyncio.gather` fires all API requests simultaneously instead of one-at-a-time
- `max_tokens=20` — a 5-word label never needs more; cuts per-request latency ~60%
- `temperature=0` — disables sampling overhead, deterministic output
- `set_topic_labels` injects labels directly, skipping the full c-TF-IDF rebuild that `update_topics` triggers

In [22]:
# pip install ollama
import ollama
import time

N_WORDS = 8
N_SAMPLE_DOCS = 10

def label_topics_ollama(topic_model, df, model="llama3.2", concurrency=4):
    """Label all BERTopic topics using a local Ollama model."""
    from concurrent.futures import ThreadPoolExecutor, as_completed

    # Build topic → representative docs lookup
    topic_docs = (
        df_train.groupby('topic_id')['text_clean']
        .apply(lambda x: x.head(N_SAMPLE_DOCS).tolist())
        .to_dict()
    )

    real_topics = [t for t in topic_model.get_topics().keys() if t != -1]
    print(f'  Labelling {len(real_topics)} topics locally with {model}...')

    def label_one(topic_id):
        words       = topic_model.get_topic(topic_id)
        keywords    = ', '.join([w for w, _ in words[:N_WORDS]])
        docs_sample = '\n'.join(topic_docs.get(topic_id, [])[:N_SAMPLE_DOCS])

        prompt = (
            f"Documents:\n{docs_sample}\n\n"
            f"Keywords: {keywords}\n\n"
            "Give a 5-word max label focusing on virality drivers "
            "(emotion, controversy, humor, trends). "
            "Output format — topic: <label>"
        )

        try:
            response = ollama.chat(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                options={"num_predict": 20, "temperature": 0},
            )
            raw   = response["message"]["content"].strip()
            label = raw.replace("topic:", "").strip()
            return topic_id, label
        except Exception as e:
            print(f'  ⚠️  Topic {topic_id} failed: {e}')
            return topic_id, None

    labels = {}
    # ThreadPoolExecutor lets multiple topics run concurrently on CPU
    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = {executor.submit(label_one, t): t for t in real_topics}
        done    = 0
        for future in as_completed(futures):
            tid, lbl = future.result()
            if lbl:
                labels[tid] = lbl
            done += 1
            if done % 10 == 0:
                print(f'  Progress: {done}/{len(real_topics)}')

    return labels


# ── Run ───────────────────────────────────────────────────────────────────────
t0 = time.time()
print("🔤 Running local Ollama topic labelling...")

topic_labels = label_topics_ollama(
    topic_model,
    df,
    model="llama3.2",
    concurrency=4,         # raise to 8 if CPU has many cores
)

if topic_labels:
    topic_model.set_topic_labels(topic_labels)
    elapsed = time.time() - t0
    print(f'\n✅ {len(topic_labels)} topics labelled in {elapsed:.1f}s')
    print('\nSample labels:')
    for tid, lbl in list(topic_labels.items())[:5]:
        print(f'  Topic {tid:>3}: {lbl}')
else:
    print('\n❌ All topics failed — is Ollama running? Try: ollama serve')

🔤 Running local Ollama topic labelling...
  Labelling 42 topics locally with llama3.2...
  Progress: 10/42
  Progress: 20/42
  Progress: 30/42
  Progress: 40/42

✅ 42 topics labelled in 2211.0s

Sample labels:
  Topic   2: Military Life Emotionally Charged
  Topic   3: Topic: "Love in the Time Chaos"
  Topic   1: Church Controversy Sparks Global Debate
  Topic   0: Politics and Election Controversy
  Topic   4: Emotion and Controversy Drives Virality


## 📊 9. Evaluate & Inspect Topics

In [31]:
# ── Topic summary: avg & std of retweets AND likes, all topics ────────────────
topic_info = topic_model.get_topic_info()

stats_by_topic = (
    df_train.groupby('topic_id')
    .agg(
        avg_retweets=('retweets', 'mean'),
        std_retweets=('retweets', 'std'),
        avg_likes=('likes', 'mean'),
        std_likes=('likes', 'std'),
        doc_count=('retweets', 'count'),
    )
    .reset_index()
    .rename(columns={'topic_id': 'Topic'})
)

# Round to 1 decimal for readability
for col in ['avg_retweets', 'std_retweets', 'avg_likes', 'std_likes']:
    stats_by_topic[col] = stats_by_topic[col].round(1)

topic_info = topic_info.merge(stats_by_topic, on='Topic', how='left')
topic_info = topic_info.sort_values(
    ['avg_retweets', 'avg_likes'],
    ascending=[False, True]
).reset_index(drop=True)

total_topics = len(topic_info)
print(f'📊 All {total_topics} topics — sorted by avg retweets (desc), avg likes (asc):')
display_df = topic_info[[
    'Topic', 'Name', 'doc_count',
    'avg_retweets', 'std_retweets',
    'avg_likes',   'std_likes',
]].copy()
display_df.insert(0, 'S.No', range(1, len(display_df) + 1))

display_df

📊 All 43 topics — sorted by avg retweets (desc), avg likes (asc):


,S.No,Topic,Name,doc_count,avg_retweets,std_retweets,avg_likes,std_likes
0,1,25,25_hair_color_source_before herself,19,61.4,25.0,44.7,26.1
1,2,21,21_employee_executive_yard record_tree skin,20,55.7,25.1,52.4,28.6
2,3,7,7_water_break_born_beat,115,55.7,28.2,53.1,26.0
3,4,33,33_this wish_authority treatment_picture form_...,16,55.4,27.6,61.6,22.9
4,5,14,14_south_really_about_security,34,55.2,31.1,47.1,29.0
5,6,19,19_teacher_phone_toward_coach,25,55.2,29.0,51.6,22.8
6,7,11,11_kid_compare_worker_wear,46,55.1,30.3,47.1,33.4
7,8,17,17_staff land_large late_its beat_put fight,29,55.0,31.1,45.5,28.7
8,9,16,16_skin_phone_hair_however,31,53.6,30.7,48.9,24.0
9,10,40,40_weight_soldier camera_at water_focus fast,12,52.5,23.1,58.8,29.3


In [32]:
# ── Per-topic keywords ────────────────────────────────────────────────────────
print('Top keywords per topic (top 5 by avg retweets):\n')
for topic_id in topic_info['Topic'].head(5):
    if topic_id == -1:
        continue
    words = topic_model.get_topic(topic_id)
    keywords = ', '.join([w for w, _ in words[:8]])
    avg_rt = topic_info.loc[topic_info['Topic'] == topic_id, 'avg_retweets'].values[0]
    print(f'  Topic {topic_id:>3} | avg retweets: {avg_rt:>8.1f} | {keywords}')

Top keywords per topic (top 5 by avg retweets):

  Topic  25 | avg retweets:     61.4 | hair, color, source, before herself, environment everyone, increase design, total far, begin phone
  Topic  21 | avg retweets:     55.7 | employee, executive, yard record, tree skin, step win, onto plant, argue like, believe to
  Topic   7 | avg retweets:     55.7 | water, break, born, beat, to, care, character, decide
  Topic  33 | avg retweets:     55.4 | this wish, authority treatment, picture form, difficult situation, say letter, control mouth, leader need, heart eat
  Topic  14 | avg retweets:     55.2 | south, really, about, security, effect, ground, big board, clearly recently


## 📐 9b. Test-Set Evaluation

Evaluate the trained BERTopic model on the **held-out test split**.

1. **Transform** test documents → topic assignments using the fitted model.
2. **Retweet prediction (RMSE):** Each document is assigned the *average retweet count* of its training-set topic. Noise docs (`topic_id == -1`) fall back to the global training mean.
3. **Popularity classification (Accuracy & F1):** Each document is assigned the *majority popularity class* of its training-set topic as the predicted label.

> RMSE measures regression error in absolute retweet counts; weighted F1 handles class imbalance.

In [34]:
from sklearn.metrics import mean_squared_error

# ── Step 1: Transform test documents ──────────────────────────────────────────
print('🔄 Transforming test documents...')
topics_test, _ = topic_model.transform(docs_test, embeddings=embeddings_test)
df_test['topic_id'] = topics_test
print(f'   Test docs transformed: {len(topics_test)}')
n_noise_test = sum(t == -1 for t in topics_test)
print(f'   Noise docs in test (-1): {n_noise_test} ({100*n_noise_test/len(topics_test):.1f}%)')

# ── Step 2: Build per-topic lookup tables from TRAIN data ─────────────────────
topic_stats = (
    df_train[df_train['topic_id'] != -1]
    .groupby('topic_id')
    .agg(
        avg_retweets=('retweets', 'mean'),
        avg_likes=('likes',    'mean'),
    )
    .to_dict(orient='index')
)
global_mean_retweets = df_train['retweets'].mean()
global_mean_likes    = df_train['likes'].mean()

# ── Step 3: Generate predictions for test set ─────────────────────────────────
df_test['pred_retweets'] = df_test['topic_id'].map(
    lambda t: topic_stats[t]['avg_retweets'] if t in topic_stats else global_mean_retweets
)
df_test['pred_likes'] = df_test['topic_id'].map(
    lambda t: topic_stats[t]['avg_likes'] if t in topic_stats else global_mean_likes
)

# ── Step 4: Compute metrics ────────────────────────────────────────────────────
rmse_retweets = np.sqrt(mean_squared_error(df_test['retweets'], df_test['pred_retweets']))
rmse_likes    = np.sqrt(mean_squared_error(df_test['likes'],    df_test['pred_likes']))

print()
print('=' * 52)
print('📐 TEST-SET EVALUATION RESULTS')
print('=' * 52)
print(f'  Retweet Prediction  — RMSE : {rmse_retweets:>8.3f}')
print(f'  Likes Prediction    — RMSE : {rmse_likes:>8.3f}')
print('=' * 52)

# ── Step 5: Predict on a new input tweet ──────────────────────────────────────
def predict_tweet(tweet_text: str) -> None:
    cleaned = clean_text(tweet_text)
    emb     = embedding_model.encode([cleaned], normalize_embeddings=True)
    tid, _  = topic_model.transform([cleaned], embeddings=emb)
    tid     = int(tid[0])

    stats   = topic_stats.get(tid, {})
    pred_rt = round(stats.get('avg_retweets', global_mean_retweets))
    pred_lk = round(stats.get('avg_likes',    global_mean_likes))

    topic_name = topic_model.get_topic_info().set_index('Topic').loc[tid, 'Name'] \
                 if tid in topic_model.get_topic_info()['Topic'].values else 'Outlier / Noise'

    print()
    print('=' * 52)
    print('🐦 TWEET PREDICTION')
    print('=' * 52)
    print(f'  Input          : {tweet_text[:80]}')
    print(f'  Topic ID       : {tid}')
    print(f'  Topic Name     : {topic_name}')
    print(f'  Pred. Retweets : {pred_rt:,}')
    print(f'  Pred. Likes    : {pred_lk:,}')
    print('=' * 52)

# Example — replace with any tweet you want to test
predict_tweet("Breaking news: scientists discover new treatment for cancer!")

2026-03-21 13:05:28,326 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.


🔄 Transforming test documents...


2026-03-21 13:05:32,172 - BERTopic - Dimensionality - Completed ✓
2026-03-21 13:05:32,174 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-21 13:05:32,355 - BERTopic - Cluster - Completed ✓


   Test docs transformed: 1982
   Noise docs in test (-1): 1786 (90.1%)

📐 TEST-SET EVALUATION RESULTS
  Retweet Prediction  — RMSE :   28.984
  Likes Prediction    — RMSE :   28.868


2026-03-21 13:05:32,977 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-21 13:05:33,043 - BERTopic - Dimensionality - Completed ✓
2026-03-21 13:05:33,053 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-21 13:05:33,062 - BERTopic - Cluster - Completed ✓



🐦 TWEET PREDICTION
  Input          : Breaking news: scientists discover new treatment for cancer!
  Topic ID       : -1
  Topic Name     : -1_provide_son_smile_work
  Pred. Retweets : 50
  Pred. Likes    : 50


## 📈 10. Visualisations

In [35]:
# ── Topic word scores ─────────────────────────────────────────────────────────
topic_model.visualize_barchart(top_n_topics=10, n_words=8)

In [27]:
# ── Topic similarity heatmap ──────────────────────────────────────────────────
topic_model.visualize_heatmap(top_n_topics=20)

In [28]:
# ── Intertopic distance map ───────────────────────────────────────────────────
topic_model.visualize_topics()

In [29]:
# ── Document-level topic map (sample 5000 docs to keep it fast) ───────────────
docs_list = list(docs_train)
embeddings_array = np.array(embeddings_train)

topic_model.visualize_documents(
    docs_list,
    embeddings=embeddings_array,
    sample=min(5000, len(docs_list)) / len(docs_list),
    hide_annotations=True,
)

## 💾 11. Save Model

In [36]:
SAVE_PATH = '../data/bertopic_model'

# ── Attach topic_pop to model before saving ───────────────────────────────────
topic_model.save(
    SAVE_PATH,
    serialization='pickle',
    save_ctfidf=True,
    save_embedding_model=EMBED_MODEL,
)
print(f"✅ Model saved to '{SAVE_PATH}'")

# To reload:
# loaded_model = BERTopic.load(SAVE_PATH)
# topic_pop_loaded = loaded_model.topic_pop_stats_

2026-03-21 13:05:59,675 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✅ Model saved to '../data/bertopic_model'
